# Project Name: Gradient Boosting Model - Home Credit Default Risk 
Eileen QIN Yuxuan, MMath Keble College, Oxford University

Target: Build a supervised machine learning pipeline to predict loan default risk using the Home 
Credit Default Risk Kaggle dataset (Data source: https://www.kaggle.com/c/home-credit-default-risk/data?utm_source=chatgpt.com ). The target is binary, and the main evaluation metric should be AUC.

    -Binary classification（二元分类）：Breaching of Contract? 0- False  1- True
    -AUC: ROC Curve:
        Horizontal axis: False positive rate (FPR)
        Vertical axis: True positive rate (TPR)

        AUC (Area Under Curve):
        The area under the ROC curve, measuring the model's ability to distinguish between positive and negative samples.
    -Feature engineering: Creating new variables (features) from raw data (i.e. from DoB to Age) to help the model learn patterns more effectively (Better features often improve performance more than choosing a different model.)
    -Missing‑Value Handling: Real datasets always have missing values. Models cannot train on NaNs, it's important to learn how to convert NaNs to reasonable values (i.e. replace NaN by median)
    -Categorical Encoding: convert category in strings to numbers, for better handling of ML models.

In [3]:
# Import and Configuration

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool

import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look nicer
sns.set(style="whitegrid", font_scale=1.1)

# For reproducibility
RANDOM_STATE = 42  # 机器学习界的一个“传统梗”，来自《银河系漫游指南》：42 是“生命、宇宙以及一切问题的终极答案”。
np.random.seed(RANDOM_STATE)

In [ ]:
# Load the Home Credit Default Risk dataset downloaded from Kaggle, username: qineileen (google logon required)
DATA_DIR = "../data/home_credit_default_risk/"
train_path = os.path.join(DATA_DIR, "application_train.csv")
# Read the main training file
df = pd.read_csv(train_path)

print("Shape of raw training data:", df.shape)
print("Columns:", len(df.columns))
print(df[["TARGET"]].head())

Shape of raw training data: (307511, 122)
Columns: 122
   TARGET
0       1
1       0
2       0
3       0
4       0


In [5]:
# 3. Preprocessing helpers

TARGET_COL = "TARGET"

# Separate features and target
y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

# Identify categorical and numeric columns
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Number of numeric features:", len(num_cols))
print("Number of categorical features:", len(cat_cols))
print("Example categorical columns:", cat_cols[:10])

Number of numeric features: 105
Number of categorical features: 16
Example categorical columns: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2240\2211429326.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
